In [1]:
import requests
import pandas as pd
from getpass import getpass

API_TOKEN = getpass("Enter your ACN API token: ")

url = "https://ev.caltech.edu/api/v1/sessions/caltech"

response = requests.get(
    url,
    auth=(API_TOKEN, "")
)

print("Status Code:", response.status_code)

if response.status_code == 200:
    print("ACN API connection successful.")
else:
    print("API request failed.")
    print(response.text[:500])

Enter your ACN API token:  ········


Status Code: 200
ACN API connection successful.


In [2]:
data = response.json()

print(data.keys())

dict_keys(['_items', '_links', '_meta'])


In [3]:
sessions = data["_items"]

print("Number of sessions:", len(sessions))

Number of sessions: 25


In [4]:
print(sessions[0].keys())

dict_keys(['_id', 'userInputs', 'sessionID', 'stationID', 'spaceID', 'siteID', 'clusterID', 'connectionTime', 'disconnectTime', 'kWhDelivered', 'doneChargingTime', 'timezone', 'userID'])


In [5]:
import pandas as pd

df = pd.DataFrame(sessions)

print("Shape:", df.shape)

df.head()

Shape: (25, 13)


,_id,userInputs,sessionID,stationID,spaceID,siteID,clusterID,connectionTime,disconnectTime,kWhDelivered,doneChargingTime,timezone,userID
0,5bc90cb9f9af8b0d7fe77cd2,None,2_39_78_362_2018-04-25 11:08:04.400812,2-39-78-362,CA-496,0002,0039,"Wed, 25 Apr 2018 11:08:04 GMT","Wed, 25 Apr 2018 13:20:10 GMT",7.932,"Wed, 25 Apr 2018 13:21:10 GMT",America/Los_Angeles,None
1,5bc90cb9f9af8b0d7fe77cd3,None,2_39_95_27_2018-04-25 13:45:09.617470,2-39-95-27,CA-319,0002,0039,"Wed, 25 Apr 2018 13:45:10 GMT","Thu, 26 Apr 2018 00:56:16 GMT",10.013,"Wed, 25 Apr 2018 16:44:15 GMT",America/Los_Angeles,None
2,5bc90cb9f9af8b0d7fe77cd4,None,2_39_79_380_2018-04-25 13:45:49.962001,2-39-79-380,CA-489,0002,0039,"Wed, 25 Apr 2018 13:45:50 GMT","Wed, 25 Apr 2018 23:04:45 GMT",5.257,"Wed, 25 Apr 2018 14:51:44 GMT",America/Los_Angeles,None
3,5bc90cb9f9af8b0d7fe77cd5,None,2_39_79_379_2018-04-25 14:37:06.460772,2-39-79-379,CA-327,0002,0039,"Wed, 25 Apr 2018 14:37:06 GMT","Wed, 25 Apr 2018 23:55:34 GMT",5.177,"Wed, 25 Apr 2018 16:05:22 GMT",America/Los_Angeles,None
4,5bc90cb9f9af8b0d7fe77cd6,None,2_39_79_381_2018-04-25 14:40:33.638896,2-39-79-381,CA-490,0002,0039,"Wed, 25 Apr 2018 14:40:34 GMT","Wed, 25 Apr 2018 23:03:12 GMT",10.119,"Wed, 25 Apr 2018 17:40:30 GMT",America/Los_Angeles,None


In [6]:
print(data["_meta"])

{'page': 1, 'max_results': 25, 'total': 31424}


## API Fields Identified

The ACN API returns the following charging-session fields:

- `_id` – Unique record identifier
- `sessionID` – Charging session identifier
- `stationID` – EV charging station identifier
- `spaceID` – Charging space identifier
- `siteID` – Charging site identifier
- `clusterID` – Charging station cluster identifier
- `connectionTime` – Time when the vehicle was connected
- `disconnectTime` – Time when the vehicle was disconnected
- `kWhDelivered` – Energy delivered during the charging session
- `doneChargingTime` – Time when charging was completed
- `timezone` – Timezone associated with the session
- `userID` – User identifier, when available
- `userInputs` – User-provided session information, when available

## Issue #1: API Connection Summary

- **Data Source:** ACN-Data
- **Access Method:** REST API
- **API Endpoint:** ACN Caltech charging sessions endpoint
- **Authentication:** API token authentication
- **Connection Status:** Successfully verified
- **Initial Records Received:** 25
- **Response Structure:** Successfully verified
- **Charging Session Fields:** Successfully identified
- **Pagination:** Will be implemented in Issue #2

In [7]:
# Issue #2: Collect EV charging sessions using API pagination

import requests
import pandas as pd
import time

all_sessions = []
page = 1
max_results = 100

while True:
    response = requests.get(
        url,
        params={
            "page": page,
            "max_results": max_results
        },
        auth=(API_TOKEN, "")
    )

    if response.status_code != 200:
        print("API request failed on page:", page)
        print("Status Code:", response.status_code)
        print(response.text[:500])
        break

    page_data = response.json()
    sessions = page_data.get("_items", [])

    if not sessions:
        break

    all_sessions.extend(sessions)

    total = page_data.get("_meta", {}).get("total", "Unknown")

    print(
        f"Page {page}: {len(sessions)} records collected | "
        f"Total collected: {len(all_sessions)} | "
        f"API total: {total}"
    )

    if len(all_sessions) >= total:
        break

    page += 1
    time.sleep(0.2)

print("\nTotal sessions collected:", len(all_sessions))

Page 1: 100 records collected | Total collected: 100 | API total: 31424
Page 2: 100 records collected | Total collected: 200 | API total: 31424
Page 3: 100 records collected | Total collected: 300 | API total: 31424
Page 4: 100 records collected | Total collected: 400 | API total: 31424
Page 5: 100 records collected | Total collected: 500 | API total: 31424
Page 6: 100 records collected | Total collected: 600 | API total: 31424
Page 7: 100 records collected | Total collected: 700 | API total: 31424
Page 8: 100 records collected | Total collected: 800 | API total: 31424
Page 9: 100 records collected | Total collected: 900 | API total: 31424
Page 10: 100 records collected | Total collected: 1000 | API total: 31424
Page 11: 100 records collected | Total collected: 1100 | API total: 31424
Page 12: 100 records collected | Total collected: 1200 | API total: 31424
Page 13: 100 records collected | Total collected: 1300 | API total: 31424
Page 14: 100 records collected | Total collected: 1400 |

In [8]:
# Issue #2: Continue EV charging session collection

import requests
import pandas as pd
import time

# Keep the 7200 records already collected
print("Existing records:", len(all_sessions))

page = 73
max_results = 100

while True:

    response = requests.get(
        url,
        params={
            "page": page,
            "max_results": max_results
        },
        auth=(API_TOKEN, ""),
        timeout=60
    )

    if response.status_code != 200:

        print(
            f"Page {page}: HTTP {response.status_code} "
            f"→ retrying in 10 seconds..."
        )

        time.sleep(10)
        continue

    page_data = response.json()
    sessions = page_data.get("_items", [])

    if not sessions:
        break

    all_sessions.extend(sessions)

    total = page_data.get("_meta", {}).get("total", 31424)

    print(
        f"Page {page}: {len(sessions)} records collected | "
        f"Total collected: {len(all_sessions)} | "
        f"API total: {total}"
    )

    if len(all_sessions) >= total:
        break

    page += 1

    # Small delay between successful requests
    time.sleep(1)

print("\nTotal sessions collected:", len(all_sessions))

Existing records: 7200
Page 73: 100 records collected | Total collected: 7300 | API total: 31424
Page 74: 100 records collected | Total collected: 7400 | API total: 31424
Page 75: 100 records collected | Total collected: 7500 | API total: 31424
Page 76: 100 records collected | Total collected: 7600 | API total: 31424
Page 77: 100 records collected | Total collected: 7700 | API total: 31424
Page 78: 100 records collected | Total collected: 7800 | API total: 31424
Page 79: 100 records collected | Total collected: 7900 | API total: 31424
Page 80: HTTP 502 → retrying in 10 seconds...
Page 80: 100 records collected | Total collected: 8000 | API total: 31424
Page 81: HTTP 502 → retrying in 10 seconds...
Page 81: 100 records collected | Total collected: 8100 | API total: 31424
Page 82: HTTP 502 → retrying in 10 seconds...
Page 82: 100 records collected | Total collected: 8200 | API total: 31424
Page 83: HTTP 502 → retrying in 10 seconds...
Page 83: 100 records collected | Total collected: 830

KeyboardInterrupt: 

In [9]:
# Issue #2: Continue collection with automatic failed-page tracking

import requests
import pandas as pd
import time

max_results = 100
total = 31424

failed_pages = []

# Continue from the page where the previous run stopped
page = 93

while page <= 315:

    success = False

    # Try each page a maximum of 5 times
    for attempt in range(1, 6):

        try:
            response = requests.get(
                url,
                params={
                    "page": page,
                    "max_results": max_results
                },
                auth=(API_TOKEN, ""),
                timeout=60
            )

            if response.status_code == 200:

                page_data = response.json()
                sessions = page_data.get("_items", [])

                if not sessions:
                    print(f"Page {page}: No records returned.")
                    success = True
                    break

                all_sessions.extend(sessions)

                print(
                    f"Page {page}: {len(sessions)} records collected | "
                    f"Total collected: {len(all_sessions)} / {total}"
                )

                success = True
                break

            else:

                print(
                    f"Page {page}: HTTP {response.status_code} "
                    f"(attempt {attempt}/5)"
                )

        except requests.exceptions.RequestException as e:

            print(
                f"Page {page}: {type(e).__name__} "
                f"(attempt {attempt}/5)"
            )

        time.sleep(10)

    # If page still failed, remember it and MOVE ON
    if not success:

        failed_pages.append(page)

        print(
            f"Page {page}: FAILED after 5 attempts "
            f"→ moving to next page"
        )

    # Save every 10 pages
    if page % 10 == 0:

        progress_df = pd.DataFrame(all_sessions)

        progress_df.to_csv(
            "acn_raw_sessions_progress.csv",
            index=False
        )

        print(
            f"Progress saved: {len(progress_df)} records"
        )

    page += 1

    # Small delay before next page
    time.sleep(2)


print("\n================================")
print("INITIAL COLLECTION COMPLETED")
print("================================")

print("Total records collected:", len(all_sessions))
print("Failed pages:", failed_pages)
print("Number of failed pages:", len(failed_pages))

Page 93: 100 records collected | Total collected: 9300 / 31424
Page 94: 100 records collected | Total collected: 9400 / 31424
Page 95: 100 records collected | Total collected: 9500 / 31424
Page 96: 100 records collected | Total collected: 9600 / 31424
Page 97: 100 records collected | Total collected: 9700 / 31424
Page 98: 100 records collected | Total collected: 9800 / 31424
Page 99: 100 records collected | Total collected: 9900 / 31424
Page 100: 100 records collected | Total collected: 10000 / 31424
Progress saved: 10000 records
Page 101: 100 records collected | Total collected: 10100 / 31424
Page 102: 100 records collected | Total collected: 10200 / 31424
Page 103: 100 records collected | Total collected: 10300 / 31424
Page 104: 100 records collected | Total collected: 10400 / 31424
Page 105: HTTP 502 (attempt 1/5)
Page 105: HTTP 502 (attempt 2/5)
Page 105: HTTP 502 (attempt 3/5)
Page 105: HTTP 502 (attempt 4/5)
Page 105: HTTP 502 (attempt 5/5)
Page 105: FAILED after 5 attempts → mov

KeyboardInterrupt: 

In [10]:
import requests

response = requests.get(
    url,
    params={
        "page": 1,
        "max_results": 100
    },
    auth=(API_TOKEN, ""),
    timeout=30
)

print("Status:", response.status_code)

if response.status_code == 200:
    test_data = response.json()

    print("\nMETA:")
    print(test_data.get("_meta"))

    print("\nLINKS:")
    print(test_data.get("_links"))
else:
    print(response.text[:500])

Status: 200

META:
{'page': 1, 'max_results': 100, 'total': 31424}

LINKS:
{'parent': {'title': 'home', 'href': '/'}, 'self': {'title': 'sessions/caltech', 'href': 'sessions/caltech?max_results=100'}, 'next': {'title': 'next page', 'href': 'sessions/caltech?max_results=100&page=2'}, 'last': {'title': 'last page', 'href': 'sessions/caltech?max_results=100&page=315'}}


In [11]:
# Issue #2: Resume EV charging session collection from Page 105
# Uses the API-provided "next" link

import requests
import pandas as pd
import time

# IMPORTANT:
# Do NOT use all_sessions = []
# We are keeping the existing 10,400 records.

print("Existing records:", len(all_sessions))

# Page 105 was the first page that failed
next_url = "https://ev.caltech.edu/api/v1/sessions/caltech?max_results=100&page=105"

total = 31424

while next_url:

    success = False

    # Retry the current page up to 5 times
    for attempt in range(1, 6):

        try:
            response = requests.get(
                next_url,
                auth=(API_TOKEN, ""),
                timeout=60
            )

            if response.status_code == 200:

                page_data = response.json()
                sessions = page_data.get("_items", [])

                if not sessions:
                    print("No more records returned.")
                    next_url = None
                    success = True
                    break

                all_sessions.extend(sessions)

                current_page = page_data.get("_meta", {}).get(
                    "page", "Unknown"
                )

                print(
                    f"Page {current_page}: "
                    f"{len(sessions)} records | "
                    f"Total collected: "
                    f"{len(all_sessions)} / {total}"
                )

                # Get the NEXT URL directly from the API
                next_link = page_data.get("_links", {}).get("next")

                if next_link:
                    next_url = (
                        "https://ev.caltech.edu/api/v1/"
                        + next_link["href"]
                    )
                else:
                    next_url = None

                success = True
                break

            else:

                print(
                    f"Page request failed: HTTP "
                    f"{response.status_code} "
                    f"(attempt {attempt}/5)"
                )

        except requests.exceptions.RequestException as e:

            print(
                f"Request error: {type(e).__name__} "
                f"(attempt {attempt}/5)"
            )

        time.sleep(15)

    # If all 5 attempts fail, stop safely
    # without losing the records already collected.
    if not success:

        print("\nCurrent page could not be collected.")
        print("Stopping safely.")
        break

    # Save progress every 10 pages
    if current_page != "Unknown" and current_page % 10 == 0:

        progress_df = pd.DataFrame(all_sessions)

        progress_df.to_csv(
            "acn_raw_sessions_progress.csv",
            index=False
        )

        print(
            f"Progress saved: {len(progress_df)} records"
        )

    time.sleep(2)


print("\n================================")
print("RESUME COLLECTION FINISHED")
print("================================")
print("Total records in memory:", len(all_sessions))

Existing records: 10400
Page 105: 100 records | Total collected: 10500 / 31424
Page 106: 100 records | Total collected: 10600 / 31424
Page 107: 100 records | Total collected: 10700 / 31424
Page 108: 100 records | Total collected: 10800 / 31424
Page 109: 100 records | Total collected: 10900 / 31424
Page 110: 100 records | Total collected: 11000 / 31424
Progress saved: 11000 records
Page request failed: HTTP 502 (attempt 1/5)
Page request failed: HTTP 502 (attempt 2/5)
Page request failed: HTTP 502 (attempt 3/5)
Page request failed: HTTP 502 (attempt 4/5)
Page request failed: HTTP 502 (attempt 5/5)

Current page could not be collected.
Stopping safely.

RESUME COLLECTION FINISHED
Total records in memory: 11000


In [12]:
import time
import requests

page = 111
max_results = 100

while True:

    try:
        response = requests.get(
            url,
            params={
                "page": page,
                "max_results": max_results
            },
            auth=(API_TOKEN, ""),
            timeout=90
        )

        if response.status_code == 200:

            page_data = response.json()
            sessions = page_data.get("_items", [])

            all_sessions.extend(sessions)

            print(
                f"Page {page}: {len(sessions)} records | "
                f"Total collected: {len(all_sessions)} / 31424"
            )

            break

        else:
            print(
                f"Page {page}: HTTP {response.status_code}"
            )
            print("Waiting 30 seconds before retry...")
            time.sleep(30)

    except requests.exceptions.RequestException as e:

        print(f"Page {page}: {type(e).__name__}")
        print("Waiting 30 seconds before retry...")
        time.sleep(30)

Page 111: 100 records | Total collected: 11100 / 31424


In [13]:
import time
import requests

page = 112
max_results = 100

while True:

    try:
        response = requests.get(
            url,
            params={
                "page": page,
                "max_results": max_results
            },
            auth=(API_TOKEN, ""),
            timeout=90
        )

        if response.status_code == 200:

            page_data = response.json()
            sessions = page_data.get("_items", [])

            all_sessions.extend(sessions)

            print(
                f"Page {page}: {len(sessions)} records | "
                f"Total collected: {len(all_sessions)} / 31424"
            )

            # Save every 10th page
            if page % 10 == 0:
                pd.DataFrame(all_sessions).to_csv(
                    "acn_raw_sessions_progress.csv",
                    index=False
                )
                print("Progress saved:", len(all_sessions))

            break

        else:
            print(
                f"Page {page}: HTTP {response.status_code}"
            )
            print("Waiting 30 seconds before retry...")
            time.sleep(30)

    except requests.exceptions.RequestException as e:

        print(f"Page {page}: {type(e).__name__}")
        print("Waiting 30 seconds before retry...")
        time.sleep(30)

Page 112: 100 records | Total collected: 11200 / 31424


In [14]:
# Issue #2: Continue collection from Page 113 to the end

import requests
import pandas as pd
import time

max_results = 100
total = 31424

failed_pages = []

print("Existing records:", len(all_sessions))
print("Starting from Page 113...\n")

for page in range(113, 316):

    success = False

    for attempt in range(1, 6):

        try:
            response = requests.get(
                url,
                params={
                    "page": page,
                    "max_results": max_results
                },
                auth=(API_TOKEN, ""),
                timeout=90
            )

            if response.status_code == 200:

                page_data = response.json()
                sessions = page_data.get("_items", [])

                if not sessions:
                    print(f"Page {page}: No records returned.")
                    success = True
                    break

                all_sessions.extend(sessions)

                print(
                    f"Page {page}: {len(sessions)} records | "
                    f"Total collected: {len(all_sessions)} / {total}"
                )

                success = True
                break

            else:

                print(
                    f"Page {page}: HTTP {response.status_code} "
                    f"(attempt {attempt}/5)"
                )

        except requests.exceptions.RequestException as e:

            print(
                f"Page {page}: {type(e).__name__} "
                f"(attempt {attempt}/5)"
            )

        time.sleep(15)

    # If page fails, remember it and continue
    if not success:

        failed_pages.append(page)

        print(
            f"Page {page}: FAILED after 5 attempts "
            f"→ moving to next page"
        )

    # Save every 10 pages
    if page % 10 == 0:

        progress_df = pd.DataFrame(all_sessions)

        progress_df.to_csv(
            "acn_raw_sessions_progress.csv",
            index=False
        )

        print(
            f"\nProgress saved: {len(progress_df)} records\n"
        )

    time.sleep(2)


print("\n====================================")
print("INITIAL COLLECTION COMPLETED")
print("====================================")

print("Total records collected:", len(all_sessions))
print("API total:", total)

print("\nFailed pages:")
print(failed_pages)

print("Number of failed pages:", len(failed_pages))

Existing records: 11200
Starting from Page 113...

Page 113: 100 records | Total collected: 11300 / 31424
Page 114: 100 records | Total collected: 11400 / 31424
Page 115: 100 records | Total collected: 11500 / 31424
Page 116: 100 records | Total collected: 11600 / 31424
Page 117: 100 records | Total collected: 11700 / 31424
Page 118: 100 records | Total collected: 11800 / 31424
Page 119: 100 records | Total collected: 11900 / 31424
Page 120: 100 records | Total collected: 12000 / 31424

Progress saved: 12000 records

Page 121: 100 records | Total collected: 12100 / 31424
Page 122: HTTP 502 (attempt 1/5)
Page 122: HTTP 502 (attempt 2/5)
Page 122: HTTP 502 (attempt 3/5)
Page 122: HTTP 502 (attempt 4/5)
Page 122: HTTP 502 (attempt 5/5)
Page 122: FAILED after 5 attempts → moving to next page
Page 123: HTTP 502 (attempt 1/5)
Page 123: HTTP 502 (attempt 2/5)
Page 123: HTTP 502 (attempt 3/5)
Page 123: HTTP 502 (attempt 4/5)
Page 123: HTTP 502 (attempt 5/5)
Page 123: FAILED after 5 attempts → 

KeyboardInterrupt: 

In [15]:
# Issue #2: Robust continuation from Page 122

import requests
import pandas as pd
import time

max_results = 100
total = 31424

failed_pages = []

print("Existing records:", len(all_sessions))
print("Resuming from Page 122...\n")

for page in range(122, 316):

    success = False
    wait_time = 10

    for attempt in range(1, 8):

        try:
            response = requests.get(
                url,
                params={
                    "page": page,
                    "max_results": max_results
                },
                auth=(API_TOKEN, ""),
                timeout=90
            )

            if response.status_code == 200:

                page_data = response.json()
                sessions = page_data.get("_items", [])

                if not sessions:
                    print(f"Page {page}: No records returned.")
                    success = True
                    break

                all_sessions.extend(sessions)

                print(
                    f"Page {page}: {len(sessions)} records | "
                    f"Total collected: {len(all_sessions)} / {total}"
                )

                success = True
                break

            else:
                print(
                    f"Page {page}: HTTP {response.status_code} "
                    f"(attempt {attempt}/7)"
                )

        except requests.exceptions.RequestException as e:
            print(
                f"Page {page}: {type(e).__name__} "
                f"(attempt {attempt}/7)"
            )

        if attempt < 7:
            print(f"Waiting {wait_time} seconds before retry...")
            time.sleep(wait_time)
            wait_time = min(wait_time * 2, 60)

    if not success:

        failed_pages.append(page)

        print(
            f"Page {page}: FAILED after 7 attempts "
            f"→ moving to next page"
        )

    # Save progress every 10 pages
    if page % 10 == 0:

        progress_df = pd.DataFrame(all_sessions)

        progress_df.to_csv(
            "acn_raw_sessions_progress.csv",
            index=False
        )

        print(
            f"\nProgress saved: {len(progress_df)} records\n"
        )

    # Small pause between pages
    time.sleep(2)


print("\n====================================")
print("COLLECTION COMPLETED")
print("====================================")

print("Total records collected:", len(all_sessions))
print("API total:", total)

print("\nFailed pages:")
print(failed_pages)

print("Number of failed pages:", len(failed_pages))

Existing records: 12100
Resuming from Page 122...

Page 122: HTTP 502 (attempt 1/7)
Waiting 10 seconds before retry...
Page 122: HTTP 502 (attempt 2/7)
Waiting 20 seconds before retry...
Page 122: HTTP 502 (attempt 3/7)
Waiting 40 seconds before retry...
Page 122: 100 records | Total collected: 12200 / 31424
Page 123: HTTP 502 (attempt 1/7)
Waiting 10 seconds before retry...
Page 123: HTTP 502 (attempt 2/7)
Waiting 20 seconds before retry...
Page 123: HTTP 502 (attempt 3/7)
Waiting 40 seconds before retry...
Page 123: 100 records | Total collected: 12300 / 31424
Page 124: HTTP 502 (attempt 1/7)
Waiting 10 seconds before retry...
Page 124: HTTP 502 (attempt 2/7)
Waiting 20 seconds before retry...
Page 124: HTTP 502 (attempt 3/7)
Waiting 40 seconds before retry...
Page 124: 100 records | Total collected: 12400 / 31424
Page 125: HTTP 502 (attempt 1/7)
Waiting 10 seconds before retry...
Page 125: HTTP 502 (attempt 2/7)
Waiting 20 seconds before retry...
Page 125: HTTP 502 (attempt 3/7)
Wai

KeyboardInterrupt: 

In [16]:
test_response = requests.get(
    url,
    params={
        "page": 1,
        "max_results": 1000
    },
    auth=(API_TOKEN, ""),
    timeout=90
)

print("Status:", test_response.status_code)

if test_response.status_code == 200:
    test_data = test_response.json()
    print("Records returned:", len(test_data.get("_items", [])))
    print("Metadata:", test_data.get("_meta"))
else:
    print(test_response.text[:500])

Status: 200
Records returned: 100
Metadata: {'page': 1, 'max_results': 100, 'total': 31424}


In [17]:
# Issue #2: Fast continuation from Page 134

import requests
import pandas as pd
import time

max_results = 100
total = 31424

failed_pages = []

print("Existing records:", len(all_sessions))
print("Starting from Page 134...\n")

for page in range(134, 316):

    success = False

    # Short retries only
    for attempt in range(1, 4):

        try:
            response = requests.get(
                url,
                params={
                    "page": page,
                    "max_results": max_results
                },
                auth=(API_TOKEN, ""),
                timeout=60
            )

            if response.status_code == 200:

                page_data = response.json()
                sessions = page_data.get("_items", [])

                if sessions:
                    all_sessions.extend(sessions)

                    print(
                        f"Page {page}: {len(sessions)} records | "
                        f"Total collected: {len(all_sessions)} / {total}"
                    )

                    success = True
                    break

            else:
                print(
                    f"Page {page}: HTTP {response.status_code} "
                    f"(attempt {attempt}/3)"
                )

        except requests.exceptions.RequestException as e:
            print(
                f"Page {page}: {type(e).__name__} "
                f"(attempt {attempt}/3)"
            )

        time.sleep(5)

    # Move on quickly if page keeps failing
    if not success:

        failed_pages.append(page)

        print(
            f"Page {page}: FAILED → moving to next page"
        )

    # Save progress every 10 pages
    if page % 10 == 0:

        progress_df = pd.DataFrame(all_sessions)

        progress_df.to_csv(
            "acn_raw_sessions_progress.csv",
            index=False
        )

        print(
            f"\nProgress saved: {len(progress_df)} records\n"
        )

    # Small pause between requests
    time.sleep(1)


print("\n====================================")
print("INITIAL COLLECTION FINISHED")
print("====================================")

print("Total records collected:", len(all_sessions))
print("API total:", total)

print("\nFailed pages:")
print(failed_pages)

print("Number of failed pages:", len(failed_pages))

Existing records: 13300
Starting from Page 134...

Page 134: 100 records | Total collected: 13400 / 31424
Page 135: 100 records | Total collected: 13500 / 31424
Page 136: 100 records | Total collected: 13600 / 31424
Page 137: 100 records | Total collected: 13700 / 31424
Page 138: 100 records | Total collected: 13800 / 31424
Page 139: 100 records | Total collected: 13900 / 31424
Page 140: 100 records | Total collected: 14000 / 31424

Progress saved: 14000 records

Page 141: 100 records | Total collected: 14100 / 31424
Page 142: 100 records | Total collected: 14200 / 31424
Page 143: 100 records | Total collected: 14300 / 31424
Page 144: HTTP 502 (attempt 1/3)
Page 144: HTTP 502 (attempt 2/3)
Page 144: HTTP 502 (attempt 3/3)
Page 144: FAILED → moving to next page
Page 145: HTTP 502 (attempt 1/3)
Page 145: HTTP 502 (attempt 2/3)
Page 145: HTTP 502 (attempt 3/3)
Page 145: FAILED → moving to next page
Page 146: HTTP 502 (attempt 1/3)
Page 146: HTTP 502 (attempt 2/3)
Page 146: HTTP 502 (attem

KeyboardInterrupt: 

In [18]:
# Save current collected records safely

current_df = pd.DataFrame(all_sessions)

current_df.to_csv(
    "acn_raw_sessions_current.csv",
    index=False
)

print("Current records saved:", len(current_df))
print("File: acn_raw_sessions_current.csv")

Current records saved: 14400
File: acn_raw_sessions_current.csv


In [19]:
# Check the current collected dataset

current_df = pd.DataFrame(all_sessions)

print("Total records:", len(current_df))
print("Number of columns:", len(current_df.columns))

print("\nColumns:")
print(current_df.columns.tolist())

print("\nDuplicate session IDs:")
print(current_df["_id"].duplicated().sum())

print("\nFirst 5 records:")
display(current_df.head())

Total records: 14400
Number of columns: 13

Columns:
['_id', 'userInputs', 'sessionID', 'stationID', 'spaceID', 'siteID', 'clusterID', 'connectionTime', 'disconnectTime', 'kWhDelivered', 'doneChargingTime', 'timezone', 'userID']

Duplicate session IDs:
0

First 5 records:


,_id,userInputs,sessionID,stationID,spaceID,siteID,clusterID,connectionTime,disconnectTime,kWhDelivered,doneChargingTime,timezone,userID
0,5bc90cb9f9af8b0d7fe77cd2,None,2_39_78_362_2018-04-25 11:08:04.400812,2-39-78-362,CA-496,0002,0039,"Wed, 25 Apr 2018 11:08:04 GMT","Wed, 25 Apr 2018 13:20:10 GMT",7.932,"Wed, 25 Apr 2018 13:21:10 GMT",America/Los_Angeles,None
1,5bc90cb9f9af8b0d7fe77cd3,None,2_39_95_27_2018-04-25 13:45:09.617470,2-39-95-27,CA-319,0002,0039,"Wed, 25 Apr 2018 13:45:10 GMT","Thu, 26 Apr 2018 00:56:16 GMT",10.013,"Wed, 25 Apr 2018 16:44:15 GMT",America/Los_Angeles,None
2,5bc90cb9f9af8b0d7fe77cd4,None,2_39_79_380_2018-04-25 13:45:49.962001,2-39-79-380,CA-489,0002,0039,"Wed, 25 Apr 2018 13:45:50 GMT","Wed, 25 Apr 2018 23:04:45 GMT",5.257,"Wed, 25 Apr 2018 14:51:44 GMT",America/Los_Angeles,None
3,5bc90cb9f9af8b0d7fe77cd5,None,2_39_79_379_2018-04-25 14:37:06.460772,2-39-79-379,CA-327,0002,0039,"Wed, 25 Apr 2018 14:37:06 GMT","Wed, 25 Apr 2018 23:55:34 GMT",5.177,"Wed, 25 Apr 2018 16:05:22 GMT",America/Los_Angeles,None
4,5bc90cb9f9af8b0d7fe77cd6,None,2_39_79_381_2018-04-25 14:40:33.638896,2-39-79-381,CA-490,0002,0039,"Wed, 25 Apr 2018 14:40:34 GMT","Wed, 25 Apr 2018 23:03:12 GMT",10.119,"Wed, 25 Apr 2018 17:40:30 GMT",America/Los_Angeles,None


In [20]:
# Issue #2: Save collected raw EV charging sessions

raw_df = pd.DataFrame(all_sessions)

raw_df.to_csv(
    "acn_raw_sessions.csv",
    index=False
)

print("Raw dataset saved successfully.")
print("Records:", len(raw_df))
print("Columns:", len(raw_df.columns))
print("File: acn_raw_sessions.csv")

Raw dataset saved successfully.
Records: 14400
Columns: 13
File: acn_raw_sessions.csv


## Issue #2: Collected EV Charging Sessions

EV charging-session data was collected from the ACN-Data API using authenticated API requests and pagination.

The API returned 31,424 available charging sessions in total. Due to intermittent API availability errors during extended collection, 14,400 unique charging sessions were successfully collected for the project.

The collected raw dataset contains 13 fields, including session ID, station ID, site ID, connection time, disconnect time, energy delivered, charging completion time, timezone, and user ID.

Duplicate checking confirmed that there are no duplicate session IDs in the collected dataset.

The raw dataset was saved as `acn_raw_sessions.csv` for subsequent data preparation and analysis.

In [2]:
# Issue #3: Load the collected raw dataset

import pandas as pd

structured_df = pd.read_csv("acn_raw_sessions.csv")

print("Dataset shape:", structured_df.shape)

print("\nColumn data types:")
print(structured_df.dtypes)

print("\nFirst 3 records:")
display(structured_df.head(3))

Dataset shape: (14400, 13)

Column data types:
_id                  object
userInputs           object
sessionID            object
stationID            object
spaceID              object
siteID                int64
clusterID             int64
connectionTime       object
disconnectTime       object
kWhDelivered        float64
doneChargingTime     object
timezone             object
userID              float64
dtype: object

First 3 records:


,_id,userInputs,sessionID,stationID,spaceID,siteID,clusterID,connectionTime,disconnectTime,kWhDelivered,doneChargingTime,timezone,userID
0,5bc90cb9f9af8b0d7fe77cd2,NaN,2_39_78_362_2018-04-25 11:08:04.400812,2-39-78-362,CA-496,2,39,"Wed, 25 Apr 2018 11:08:04 GMT","Wed, 25 Apr 2018 13:20:10 GMT",7.932,"Wed, 25 Apr 2018 13:21:10 GMT",America/Los_Angeles,NaN
1,5bc90cb9f9af8b0d7fe77cd3,NaN,2_39_95_27_2018-04-25 13:45:09.617470,2-39-95-27,CA-319,2,39,"Wed, 25 Apr 2018 13:45:10 GMT","Thu, 26 Apr 2018 00:56:16 GMT",10.013,"Wed, 25 Apr 2018 16:44:15 GMT",America/Los_Angeles,NaN
2,5bc90cb9f9af8b0d7fe77cd4,NaN,2_39_79_380_2018-04-25 13:45:49.962001,2-39-79-380,CA-489,2,39,"Wed, 25 Apr 2018 13:45:50 GMT","Wed, 25 Apr 2018 23:04:45 GMT",5.257,"Wed, 25 Apr 2018 14:51:44 GMT",America/Los_Angeles,NaN


In [3]:
# Inspect the userInputs field

print("userInputs data type:")
print(type(structured_df["userInputs"].iloc[0]))

print("\nFirst userInputs value:")
print(structured_df["userInputs"].iloc[0])

userInputs data type:
<class 'float'>

First userInputs value:
nan


In [4]:
# Issue #3: Inspect column structure

print("Column-wise non-null values:")
print(structured_df.notna().sum())

print("\nUnique values in key identifier columns:")
for col in ["_id", "sessionID", "stationID", "spaceID", "siteID", "clusterID"]:
    print(f"{col}: {structured_df[col].nunique()} unique values")

Column-wise non-null values:
_id                 14400
userInputs           1730
sessionID           14400
stationID           14400
spaceID             14400
siteID              14400
clusterID           14400
connectionTime      14400
disconnectTime      14400
kWhDelivered        14400
doneChargingTime    14394
timezone            14400
userID               1730
dtype: int64

Unique values in key identifier columns:
_id: 14400 unique values
sessionID: 14400 unique values
stationID: 54 unique values
spaceID: 54 unique values
siteID: 1 unique values
clusterID: 1 unique values


In [5]:
# Issue #3: Create structured charging-session dataset

structured_df = structured_df[
    [
        "_id",
        "sessionID",
        "userID",
        "stationID",
        "spaceID",
        "siteID",
        "clusterID",
        "connectionTime",
        "disconnectTime",
        "doneChargingTime",
        "kWhDelivered",
        "timezone",
        "userInputs"
    ]
].copy()

print("Structured dataset shape:", structured_df.shape)

print("\nStructured columns:")
print(structured_df.columns.tolist())

print("\nFirst 5 records:")
display(structured_df.head())

Structured dataset shape: (14400, 13)

Structured columns:
['_id', 'sessionID', 'userID', 'stationID', 'spaceID', 'siteID', 'clusterID', 'connectionTime', 'disconnectTime', 'doneChargingTime', 'kWhDelivered', 'timezone', 'userInputs']

First 5 records:


,_id,sessionID,userID,stationID,spaceID,siteID,clusterID,connectionTime,disconnectTime,doneChargingTime,kWhDelivered,timezone,userInputs
0,5bc90cb9f9af8b0d7fe77cd2,2_39_78_362_2018-04-25 11:08:04.400812,NaN,2-39-78-362,CA-496,2,39,"Wed, 25 Apr 2018 11:08:04 GMT","Wed, 25 Apr 2018 13:20:10 GMT","Wed, 25 Apr 2018 13:21:10 GMT",7.932,America/Los_Angeles,NaN
1,5bc90cb9f9af8b0d7fe77cd3,2_39_95_27_2018-04-25 13:45:09.617470,NaN,2-39-95-27,CA-319,2,39,"Wed, 25 Apr 2018 13:45:10 GMT","Thu, 26 Apr 2018 00:56:16 GMT","Wed, 25 Apr 2018 16:44:15 GMT",10.013,America/Los_Angeles,NaN
2,5bc90cb9f9af8b0d7fe77cd4,2_39_79_380_2018-04-25 13:45:49.962001,NaN,2-39-79-380,CA-489,2,39,"Wed, 25 Apr 2018 13:45:50 GMT","Wed, 25 Apr 2018 23:04:45 GMT","Wed, 25 Apr 2018 14:51:44 GMT",5.257,America/Los_Angeles,NaN
3,5bc90cb9f9af8b0d7fe77cd5,2_39_79_379_2018-04-25 14:37:06.460772,NaN,2-39-79-379,CA-327,2,39,"Wed, 25 Apr 2018 14:37:06 GMT","Wed, 25 Apr 2018 23:55:34 GMT","Wed, 25 Apr 2018 16:05:22 GMT",5.177,America/Los_Angeles,NaN
4,5bc90cb9f9af8b0d7fe77cd6,2_39_79_381_2018-04-25 14:40:33.638896,NaN,2-39-79-381,CA-490,2,39,"Wed, 25 Apr 2018 14:40:34 GMT","Wed, 25 Apr 2018 23:03:12 GMT","Wed, 25 Apr 2018 17:40:30 GMT",10.119,America/Los_Angeles,NaN


In [6]:
# Save structured dataset

structured_df.to_csv(
    "acn_structured_sessions.csv",
    index=False
)

print("Structured dataset saved successfully.")
print("Records:", len(structured_df))
print("Columns:", len(structured_df.columns))
print("File: acn_structured_sessions.csv")

Structured dataset saved successfully.
Records: 14400
Columns: 13
File: acn_structured_sessions.csv
